# Gemma 4-E4B 微调教程

本Notebook演示如何使用 **Unsloth** 框架微调 Google Gemma 4-E4B 模型。

## 特点

- 使用 QLoRA 4-bit量化，仅需约10GB VRAM
- 支持在免费Colab T4 GPU上运行
- 训练速度提升2x，VRAM节省60%

## 硬件需求

| 模型       | QLoRA VRAM | 推荐GPU             |
| ---------- | ---------- | ------------------- |
| E4B (4.5B) | ~10 GB     | Colab T4 / RTX 3060 |
| 31B        | ~22 GB     | RTX 4090            |

---


## 1. 环境安装

首先安装 Unsloth 及相关依赖库。


In [ ]:
# 安装 Unsloth 和相关依赖
# 注意：在 Colab 上运行时，取消下面的注释

# %%capture
# import os
# if 'COLAB_' not in ''.join(os.environ.keys()):
#     !pip install unsloth
# else:
#     !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton cut_cross_entropy unsloth_zoo
#     !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
#     !pip install --no-deps unsloth

In [ ]:
# 本地环境安装（非Colab）
try:
    import unsloth
    print(f"Unsloth 已安装，版本: {unsloth.__version__}")
except ImportError:
    print("正在安装 Unsloth...")
    import subprocess
    subprocess.run(['pip', 'install', 'unsloth'], check=True)
    import unsloth
    print(f"Unsloth 安装完成，版本: {unsloth.__version__}")

In [ ]:
# 导入必要的库
import torch
from unsloth import FastModel
from transformers import AutoTokenizer
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import json
from pathlib import Path

# 检查GPU可用性
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_memory:.1f} GB")
    print(f"CUDA版本: {torch.version.cuda}")
else:
    print("警告：未检测到GPU，训练将非常缓慢！")

## 2. 加载模型

使用 Unsloth 的 `FastModel` 加载 Gemma 4-E4B 模型，采用 4-bit QLoRA 量化。

### 模型路径配置

- **本地路径**: `/raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit` - 已下载的本地模型
- **在线路径**: `unsloth/gemma-4-E4B-it-bnb-4bit` - 从HuggingFace下载（需要网络）


In [ ]:
# 模型配置参数
# ============================================================
# 【重要】请根据您的实际环境修改以下路径配置
# ============================================================

# 方式1: 使用HuggingFace在线下载（推荐，自动下载模型）
MODEL_NAME = "unsloth/gemma-4-E4B-it-bnb-4bit"  # HuggingFace在线路径

# 方式2: 使用本地已下载的模型路径（如果已下载，可避免网络下载）
# MODEL_NAME = "/raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"  # Linux服务器路径
# MODEL_NAME = r"D:\models\gemma-4-E4B-it-unsloth-bnb-4bit"  # Windows本地路径

MAX_SEQ_LENGTH = 2048  # 最大序列长度
LOAD_IN_4BIT = True   # 使用4-bit量化（本地模型已经是4bit量化，此参数可选）

# 设置环境变量禁用HuggingFace统计信息（避免连接超时）
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'

# 检查本地模型是否存在
import os
if os.path.exists(MODEL_NAME):
    print(f"使用本地模型: {MODEL_NAME}")
    model_files = os.listdir(MODEL_NAME)
    print(f"模型文件: {model_files[:5]}...")
else:
    print(f"警告：本地模型路径不存在: {MODEL_NAME}")
    print("请修改 MODEL_NAME 为正确的路径，或使用在线路径下载模型")

# 加载模型和tokenizer
print(f"正在加载模型: {MODEL_NAME}")
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # 自动检测dtype
    device_map = "balanced",
    disable_log_stats=True,  # 禁用HuggingFace统计信息，避免连接超时
)

print("模型加载完成！")
print(f"模型参数量: {model.num_parameters() / 1e9:.2f}B")

## 3. 配置 LoRA 适配器

添加 LoRA adapters，只更新少量参数，大幅降低训练成本。

### LoRA 参数说明

| 参数           | 说明                     | 推荐值             |
| -------------- | ------------------------ | ------------------ |
| r              | LoRA秩，控制可训练参数量 | 16 (默认) 或 32-64 |
| lora_alpha     | 缩放因子                 | 通常设为 r         |
| lora_dropout   | Dropout率                | Unsloth推荐0       |
| target_modules | 目标模块                 | attention + FFN    |


In [ ]:
# 配置 LoRA
model = FastModel.get_peft_model(
    model,
    r=16,                     # LoRA秩
    lora_alpha=16,            # 缩放因子（通常等于r）
    lora_dropout=0,           # Dropout率（Unsloth推荐0）
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention层
        "gate_proj", "up_proj", "down_proj",       # FFN层
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth优化的梯度检查点
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# 打印可训练参数信息
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percent = trainable_params / total_params * 100

print(f"可训练参数: {trainable_params:,} ({trainable_percent:.2f}%)")
print(f"总参数: {total_params:,}")

## 4. 准备训练数据

支持多种数据格式：

1. **Messages格式** - Gemma 4原生对话格式
2. **Alpaca格式** - 通用指令格式
3. **自定义JSONL** - 自定义格式

### 数据格式示例

```json
{
  "messages": [
    { "role": "user", "content": "用户问题" },
    { "role": "assistant", "content": "回答" }
  ]
}
```


In [ ]:
# 数据路径配置
# ============================================================
# 【重要】训练数据路径配置
# ============================================================

# 方式1: 使用项目自带的示例数据文件（相对路径）
DATA_PATH = "./unsloth_train_data.jsonl"  # 项目自带数据文件

# 方式2: 使用自定义数据路径
# DATA_PATH = r"/path/to/your/data.jsonl"  # Linux/自定义路径

# 方式3: 使用HuggingFace数据集（备选）
# DATA_PATH = "Abirate/english_quotes"  # 示例数据集

# 示例数据（用于测试，当数据文件不存在时使用）
EXAMPLE_DATA = [
    {
        "messages": [
            {"role": "user", "content": "请解释什么是机器学习。"},
            {"role": "assistant", "content": "机器学习是人工智能的一个分支，它使计算机能够从数据中学习并做出决策或预测，而无需明确编程。主要类型包括监督学习、无监督学习和强化学习。"}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "什么是深度学习？"},
            {"role": "assistant", "content": "深度学习是机器学习的一个子领域，使用多层神经网络来处理复杂的数据模式。它在图像识别、自然语言处理和语音识别等领域表现出色。"}
        ]
    },
]

# 检查数据文件是否存在并加载
data_file = Path(DATA_PATH)
if data_file.exists():
    print(f"数据文件存在: {DATA_PATH}")
    dataset = load_dataset("json", data_files=str(data_file), split="train")
    print(f"数据集大小: {len(dataset)} 条")
    print(f"数据集列: {dataset.column_names}")
else:
    print(f"警告：数据文件不存在: {DATA_PATH}")
    print("使用示例数据进行测试...")
    temp_file = "temp_example_data.jsonl"
    with open(temp_file, 'w', encoding='utf-8') as f:
        for item in EXAMPLE_DATA:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    dataset = load_dataset("json", data_files=temp_file, split="train")
    print(f"示例数据集大小: {len(dataset)} 条")

# 显示数据样本
print("\n数据样本预览:")
sample = dataset[0]
print(json.dumps(sample, ensure_ascii=False, indent=2)[:500] + "...")

In [ ]:
# 数据预处理函数
def format_data(sample):
    """
    将数据格式化为模型可接受的格式
    """
    messages = sample.get("messages", [])
    
    # 使用tokenizer的chat template格式化
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return text

# 应用格式化
if "messages" in dataset.column_names:
    dataset = dataset.map(
        lambda x: {"text": format_data(x)},
        remove_columns=dataset.column_names
    )

print("数据预处理完成！")
print(f"处理后数据列: {dataset.column_names}")
print(f"\n格式化样本:")
print(dataset[0]['text'][:500] + "...")

## 5. 配置训练参数

使用 `SFTTrainer` 进行监督微调训练。


In [ ]:
# 训练配置参数
TRAINING_CONFIG = {
    "per_device_train_batch_size":4 ,      # 每GPU批次大小
    "gradient_accumulation_steps": 8,      # 梯度累积步数
    "warmup_ratio": 0.05,                   # 预热比例
    "num_train_epochs": 1,                  # 训练轮数
    "learning_rate": 2e-4,                  # 学习率
    "optim": "adamw_8bit",                 # 优化器
    "weight_decay": 0.01,                   # 权重衰减
    "lr_scheduler_type": "linear",         # 学习率调度器
    "seed": 3407,                           # 随机种子
    "logging_steps": 10,                    # 日志记录步数
    "save_steps": 300,                      # 保存步数
    "save_total_limit": 2,                  # 最多保存数量
    "output_dir": "outputs",               # 输出目录
    "report_to": "none",                   # 报告平台
}

# 创建训练器
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        **TRAINING_CONFIG,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
    ),
)

print("训练器创建完成！")
print(f"有效批次大小: {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")

## 6. 开始训练

执行训练过程，监控loss变化。


In [ ]:
# 显示GPU内存状态
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"初始VRAM使用: {start_gpu_memory} GB / {max_memory} GB")
print(f"剩余VRAM: {max_memory - start_gpu_memory} GB")

In [ ]:
# 开始训练
print("开始训练...")
trainer_stats = trainer.train()

print("\n训练完成！")

# 获取训练统计信息（使用.get()避免KeyError）
metrics = trainer_stats.metrics

train_runtime = metrics.get('train_runtime', 0)
train_loss = metrics.get('train_loss', 0)
train_steps = metrics.get('train_steps', 0)

# 计算实际样本数
effective_batch_size = TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']
total_samples = len(dataset) if hasattr(dataset, '__len__') else 0

print(f"训练时长: {train_runtime:.2f} 秒")
print(f"训练步数: {train_steps}")
print(f"最终loss: {train_loss:.4f}")
print(f"数据集样本数: {total_samples}")
print(f"有效批次大小: {effective_batch_size}")

# 显示所有可用的metrics
print("\n所有训练统计信息:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

In [ ]:
# 显示训练后GPU内存状态
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"训练后VRAM使用: {used_memory} GB")
print(f"LoRA占用VRAM: {used_memory_for_lora} GB")
print(f"VRAM使用率: {used_percentage}%")
print(f"LoRA VRAM占用率: {lora_percentage}%")

## 7. 推理测试

测试微调后的模型效果。


In [ ]:
# 推理测试函数
def test_inference(prompt, max_new_tokens=128):
    """
    测试模型推理
    
    Args:
        prompt: 输入提示
        max_new_tokens: 最大生成token数
    """
    # 构建输入（Gemma 4格式要求content为列表）
    messages = [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        }
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    # 生成回复
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=1.5,
        min_p=0.1,
    )
    
    # 解码输出
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return response[0]

# 测试推理
test_prompts = [
    "请解释什么是机器学习。",
    "什么是深度学习？",
    "请介绍一下神经网络。",
]

print("推理测试结果:")
print("=" * 50)
for prompt in test_prompts:
    print(f"\n问题: {prompt}")
    response = test_inference(prompt)
    print(f"回答: {response}")
    print("-" * 50)

## 8. 保存模型

提供多种保存选项：

1. **LoRA adapters** - 只保存适配器（体积小）
2. **合并模型** - 合并LoRA到基础模型
3. **GGUF格式** - 用于Ollama/llama.cpp部署


In [ ]:
# 保存LoRA adapters
# 请修改为您想要保存的路径
LORA_OUTPUT_DIR = "/raid5/sh/model/finetuned/gemma4_e4b_lora"  # Linux路径

# 或者使用Windows路径（如果在Windows上运行）
# LORA_OUTPUT_DIR = r"d:\WorkPlace\Pycharm\multimode_data_clean\gemma4_e4b_lora"

model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print(f"LoRA adapters已保存到: {LORA_OUTPUT_DIR}")

In [ ]:
# 选项：保存为GGUF格式（用于Ollama部署）
# 取消注释以执行

# GGUF_OUTPUT_DIR = "/raid5/sh/model/finetuned/gemma4_e4b_gguf"
# QUANTIZATION_METHOD = "q4_k_m"  # 可选: q4_k_m, q8_0, f16

# model.save_pretrained_gguf(
#     GGUF_OUTPUT_DIR,
#     tokenizer,
#     quantization_method=QUANTIZATION_METHOD,
# )

# print(f"GGUF模型已保存到: {GGUF_OUTPUT_DIR}")
# print(f"量化方法: {QUANTIZATION_METHOD}")

In [ ]:
# 选项：推送到Hugging Face Hub
# 取消注释以执行（需要设置HF_TOKEN）

# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")  # 替换为您的HF token

# HF_REPO_NAME = "your-username/gemma4-e4b-finetuned"  # 替换为您的repo名称
# model.push_to_hub(HF_REPO_NAME)
# tokenizer.push_to_hub(HF_REPO_NAME)

# print(f"模型已推送到: https://huggingface.co/{HF_REPO_NAME}")

## 9. 加载保存的模型进行测试

验证保存的模型是否正常工作。


In [ ]:
# 9. 从磁盘重新加载微调后的模型进行测试
# 
# Gemma 4 使用了特殊的 Gemma4ClippableLinear 模块，需要先patch PEFT才能正确加载LoRA adapter
# 参考: https://github.com/huggingface/peft/issues/3130

# 设置环境变量禁用HuggingFace统计信息（避免连接超时）
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'

# Patch PEFT以支持 Gemma4ClippableLinear
def patch_peft_for_gemma4():
    """
    修复PEFT对Gemma4ClippableLinear的支持
    """
    try:
        from peft.tuners.lora import model as lora_model
        from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear
        
        _original_create_new_module = lora_model.LoraModel._create_new_module
        
        @classmethod
        def _patched_create_new_module(cls, lora_config, adapter_name, target, **kwargs):
            # 如果是Gemma4ClippableLinear，提取内部的linear层
            if isinstance(target, Gemma4ClippableLinear):
                return _original_create_new_module.__func__(cls, lora_config, adapter_name, target.linear, **kwargs)
            return _original_create_new_module.__func__(cls, lora_config, adapter_name, target, **kwargs)
        
        lora_model.LoraModel._create_new_module = _patched_create_new_module
        print("PEFT已patch，支持Gemma4ClippableLinear")
        return True
    except Exception as e:
        print(f"Patch失败: {e}")
        print("将使用exclude_modules方式加载")
        return False

# 执行patch
patch_success = patch_peft_for_gemma4()

if os.path.exists(LORA_OUTPUT_DIR):
    print(f"\n从磁盘加载保存的模型: {LORA_OUTPUT_DIR}")
    
    # 方法1: 使用FastModel加载基础模型
    print("正在加载基础模型...")
    loaded_base_model, loaded_tokenizer = FastModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        disable_log_stats=True,
    )
    
    # 方法2: 添加LoRA adapter
    print("正在加载LoRA adapter...")
    from peft import PeftModel
    
    try:
        # 尝试直接加载（patch成功时可用）
        loaded_model = PeftModel.from_pretrained(
            loaded_base_model,
            LORA_OUTPUT_DIR,
            is_trainable=False  # 仅用于推理
        )
        print("LoRA adapter加载成功！")
    except ValueError as e:
        # 如果直接加载失败，使用exclude_modules方式
        print(f"直接加载失败，使用exclude_modules方式: {e}")
        
        from peft import LoraConfig
        
        # 先加载adapter配置
        adapter_config = LoraConfig.from_pretrained(LORA_OUTPUT_DIR)
        
        # 添加exclude_modules以避开vision_tower和audio_tower
        adapter_config.exclude_modules = ['vision_tower', 'audio_tower']
        
        loaded_model = PeftModel(
            loaded_base_model,
            adapter_config,
            adapter_name="default"
        )
        
        # 手动加载权重
        import safetensors
        adapter_weights_path = os.path.join(LORA_OUTPUT_DIR, 'adapter_model.safetensors')
        if os.path.exists(adapter_weights_path):
            with safetensors.safe_open(adapter_weights_path, framework='pt') as f:
                for key in f.keys():
                    tensor = f.get_tensor(key)
                    # 设置权重到对应的模块
                    parts = key.split('.')
                    module = loaded_model
                    for part in parts[:-1]:
                        if hasattr(module, part):
                            module = getattr(module, part)
                    if hasattr(module, parts[-1]):
                        setattr(module, parts[-1], torch.nn.Parameter(tensor))
        print("LoRA adapter权重手动加载完成")
    
    print("\n模型加载完成！开始测试...")
    
    # 测试加载的模型
    def test_loaded_model(prompt, max_new_tokens=128):
        messages = [
            {
                "role": "user",
                "content": [{"type": "text", "text": prompt}]
            }
        ]
        
        inputs = loaded_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to("cuda")
        
        outputs = loaded_model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            temperature=0.7,
            top_p=0.9,
        )
        
        response = loaded_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        return response
    
    # 测试示例
    test_prompts_reload = [
        "请解释什么是机器学习。",
        "什么是深度学习？",
    ]
    
    print("\n重新加载后的模型测试结果:")
    print("=" * 60)
    for prompt in test_prompts_reload:
        print(f"\n问题: {prompt}")
        response = test_loaded_model(prompt)
        print(f"回答: {response}")
        print("-" * 60)
    
    # 显示保存的文件信息
    print(f"\nLoRA adapters目录内容: {LORA_OUTPUT_DIR}")
    saved_files = os.listdir(LORA_OUTPUT_DIR)
    print(f"保存的文件: {saved_files}")
    
else:
    print(f"模型目录不存在: {LORA_OUTPUT_DIR}")
    print("请先运行前面的训练和保存步骤")
    print("请先运行前面的保存代码，或检查路径是否正确")

---

# 附录：分布式训练指南（单机多卡/多机多卡）

Unsloth 目前支持通过 **DDP** (Distributed Data Parallel) 和 **FSDP** (Fully Sharded Data Parallel) 进行多GPU分布式训练。

## 分布式训练方式对比

| 方式 | 适用场景                 | 显存优势           | 通信开销 |
| ---- | ------------------------ | ------------------ | -------- |
| DDP  | 单机多卡，模型可放入单卡 | 无额外优势         | 较低     |
| FSDP | 大模型，多机多卡         | 显存分片，大幅节省 | 较高     |

## 10. 分布式训练配置

分布式训练需要将Notebook转换为独立的Python脚本，使用 `torchrun` 或 `accelerate launch` 启动。


### 10.1 检查GPU数量和环境


In [ ]:
# 检查多GPU环境
import torch

num_gpus = torch.cuda.device_count()
print(f"检测到 GPU 数量: {num_gpus}")

for i in range(num_gpus):
    gpu_name = torch.cuda.get_device_name(i)
    gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {gpu_name}, VRAM: {gpu_memory:.1f} GB")

if num_gpus < 2:
    print("\n警告：检测到少于2个GPU，分布式训练需要多GPU环境")
    print("请确保：")
    print("  1. 使用 CUDA_VISIBLE_DEVICES 指定可用GPU")
    print("  2. 在多GPU服务器上运行")
else:
    print(f"\n可以使用 {num_gpus} GPU 进行分布式训练")

### 10.2 分布式训练启动命令

以下是在终端执行的命令（需要在Python脚本模式下运行）。


In [ ]:
# 分布式训练启动命令示例
# 这些命令需要在终端中执行，而非在Notebook中

# ==================== DDP 方式 ====================
# 单机多卡 DDP 训练
ddp_command = f'''torchrun --nproc_per_node={num_gpus} train_distributed.py \
    --model_name /raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit \
    --data_path /raid5/sh/data/unsloth_train_data.jsonl \
    --output_dir /raid5/sh/model/finetuned/gemma4_e4b_lora \
    --max_seq_length 2048 \
    --per_device_batch_size 2 \
    --gradient_accumulation_steps 4 \
    --learning_rate 2e-4 \
    --num_epochs 1 \
    --use_ddp'''

print("DDP 启动命令（单机多卡）:")
print(ddp_command)

# ==================== FSDP 方式 ====================
# FSDP 训练（适用于大模型或显存不足的情况）
fsdp_command = f'''torchrun --nproc_per_node={num_gpus} train_distributed.py \
    --model_name /raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit \
    --data_path /raid5/sh/data/unsloth_train_data.jsonl \
    --output_dir /raid5/sh/model/finetuned/gemma4_e4b_lora_fsdp \
    --max_seq_length 2048 \
    --per_device_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --learning_rate 2e-4 \
    --num_epochs 1 \
    --use_fsdp'''

print("\nFSDP 启动命令:")
print(fsdp_command)

# ==================== 多机多卡 ====================
# 多机多卡训练（假设2台机器，每台4卡）
multi_node_command = '''torchrun \
    --nnodes=2 \
    --nproc_per_node=4 \
    --node_rank=0 \
    --master_addr="192.168.1.1" \
    --master_port=29500 \
    train_distributed.py \
    --model_name /raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit \
    --data_path /raid5/sh/data/unsloth_train_data.jsonl \
    --output_dir /raid5/sh/model/finetuned/gemma4_e4b_lora \
    --use_fsdp'''

print("\n多机多卡启动命令（主节点 node_rank=0）:")
print(multi_node_command)

# ==================== Accelerate 方式 ====================
accelerate_command = '''accelerate launch train_distributed.py \
    --model_name /raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit \
    --data_path /raid5/sh/data/unsloth_train_data.jsonl \
    --output_dir /raid5/sh/model/finetuned/gemma4_e4b_lora'''

print("\nAccelerate 启动命令（需要先运行 accelerate config）:")
print(accelerate_command)

### 10.3 指定GPU设备

使用 `CUDA_VISIBLE_DEVICES` 环境变量指定可用GPU。


In [ ]:
# GPU设备指定示例

# 只使用GPU 0,1,2,3
print("指定GPU 0-3:")
print("CUDA_VISIBLE_DEVICES=0,1,2,3 torchrun --nproc_per_node=4 train_distributed.py ...")

# 只使用GPU 4,5,6,7
print("\n指定GPU 4-7:")
print("CUDA_VISIBLE_DEVICES=4,5,6,7 torchrun --nproc_per_node=4 train_distributed.py ...")

# 模型分片加载（适用于单卡无法加载的大模型）
print("\n模型分片加载（device_map='balanced'):")
print("# 在代码中使用 device_map='balanced' 将模型分片到多个GPU")

### 10.4 模型分片加载

对于无法在单GPU上加载的大模型，使用 `device_map="balanced"` 进行模型分片。


In [ ]:
# 模型分片加载示例代码
# 此代码用于分布式训练脚本中

model_sharding_code = '''
from unsloth import FastModel

# 模型分片到多个GPU（适用于大模型如70B）
model, tokenizer = FastModel.from_pretrained(
    model_name="/raid5/sh/model/Llama-3.3-70B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    device_map="balanced",  # 自动平衡分配到各GPU
)
'''

### 10.5 分布式训练脚本结构

分布式训练脚本 `train_distributed.py` 的基本结构如下：


In [ ]:
# 显示分布式训练脚本的基本结构

script_structure = '''
# train_distributed.py 基本结构

import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from unsloth import FastModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

def setup_distributed():
    """初始化分布式环境"""
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    torch.cuda.set_device(local_rank)
    return local_rank

def cleanup_distributed():
    """清理分布式环境"""
    dist.destroy_process_group()

def main():
    # 1. 初始化分布式
    local_rank = setup_distributed()
    
    # 2. 加载模型（每个进程加载相同的模型副本）
    model, tokenizer = FastModel.from_pretrained(...)
    model = FastModel.get_peft_model(model, ...)
    
    # 3. 用DDP包装模型
    model = DDP(model, device_ids=[local_rank])
    
    # 4. 加载数据（每个进程处理数据的一部分）
    dataset = load_dataset(...)
    
    # 5. 创建训练器并训练
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, ...)
    trainer.train()
    
    # 6. 仅在rank 0保存模型
    if dist.get_rank() == 0:
        model.module.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
    
    # 7. 清理
    cleanup_distributed()

if __name__ == "__main__":
    main()
'''

---

## 总结

### 单GPU vs 分布式训练选择指南

| 场景 | 推荐方式 |
|------|----------|
| E4B模型，单卡VRAM >= 10GB | 单GPU训练（本Notebook） |
| 31B模型，单卡VRAM不足 | FSDP分片训练 |
| 单机多卡，加速训练 | DDP训练 |
| 多机多卡，大规模训练 | FSDP + 多机 |

### 训练配置最佳实践

| 任务类型 | 最小样本数 | 推荐范围 |
|----------|------------|----------|
| 风格/语气调整 | 100-500 | 200-1,000 |
| 分类/提取 | 500-2,000 | 1,000-5,000 |
| 领域知识注入 | 1,000-5,000 | 5,000-50,000 |
| 复杂推理 | 5,000+ | 10,000-50,000 |

### 部署建议
1. **本地部署**: 使用GGUF格式 + Ollama
2. **API服务**: 使用vLLM加载LoRA adapters
3. **云端部署**: 推送到Hugging Face Hub

---

**参考资源：**

- [Unsloth多GPU文档](https://unsloth.ai/docs/basics/multi-gpu-training-with-unsloth)
- [Unsloth官方文档](https://unsloth.ai/)
- [Gemma 4模型](https://huggingface.co/google/gemma-4)
- [TRL库](https://github.com/huggingface/trl)
- [PyTorch分布式训练](https://pytorch.org/tutorials/intermediate/dist_training_tutorial.html)
